In [26]:
import json
import math

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

# NOTE: Done for only 100g dataset and maxCores 100
# TODO: Check if the buckets hold for other configs we want to use:
# maxCores=200? or maxCores=75?

# NOTE: Also hardcodes min_task_runtime to 12s. But need to verify the
# task_runtime computed here is same as loader. Loader also does rectangular
# reshaping of tasks to stage runtime.

def extract_tpch_data(json_data):
    extracted_data = {}
    
    for query_key, stages in json_data.items():
        if "100g" in query_key and "maxCores_100" in query_key:
            query_id = query_key.split("_")[1]  # Extract query number (e.g., 'q1')
            extracted_data[query_id] = [
                (stage["stage_id"], stage["num_tasks"], math.ceil(max(12000, int(stage["average_runtime_ms"]))/1000))
                for stage in stages
            ]
    
    return extracted_data

if __name__ == "__main__":
    file_path = "/home/dgarg39/erdos-scheduling-simulator/profiles/workload/tpch/cloudlab/cloudlab_22query_tpch_profiles.json"
    json_data = load_json(file_path)
    result = extract_tpch_data(json_data)
    print(json.dumps(result, indent=4))

{
    "q1": [
        [
            0,
            593,
            18
        ],
        [
            1,
            1,
            12
        ],
        [
            2,
            1,
            12
        ]
    ],
    "q2": [
        [
            0,
            100,
            14
        ],
        [
            1,
            100,
            12
        ],
        [
            2,
            100,
            12
        ],
        [
            3,
            100,
            12
        ],
        [
            4,
            146,
            12
        ],
        [
            5,
            35,
            12
        ],
        [
            6,
            35,
            12
        ],
        [
            7,
            1,
            12
        ],
        [
            8,
            1,
            12
        ],
        [
            9,
            2,
            12
        ],
        [
            10,
            1,
            12
        ],
        [
            11,
   

In [27]:
result

{'q1': [(0, 593, 18), (1, 1, 12), (2, 1, 12)],
 'q2': [(0, 100, 14),
  (1, 100, 12),
  (2, 100, 12),
  (3, 100, 12),
  (4, 146, 12),
  (5, 35, 12),
  (6, 35, 12),
  (7, 1, 12),
  (8, 1, 12),
  (9, 2, 12),
  (10, 1, 12),
  (11, 1, 12),
  (12, 13, 12),
  (13, 1, 12),
  (14, 1, 12),
  (15, 1, 12),
  (16, 1, 12)],
 'q3': [(0, 593, 18), (1, 133, 17), (2, 100, 12), (3, 139, 12), (4, 139, 12)],
 'q4': [(0, 593, 17), (1, 133, 14), (2, 1, 12), (3, 141, 12), (4, 1, 12)],
 'q5': [(0, 593, 19),
  (1, 133, 13),
  (2, 1, 12),
  (3, 200, 12),
  (4, 100, 12),
  (5, 138, 12),
  (6, 35, 12),
  (7, 140, 12),
  (8, 1, 12),
  (9, 2, 12),
  (10, 1, 12),
  (11, 1, 12),
  (12, 1, 12)],
 'q6': [(0, 593, 14), (1, 1, 12)],
 'q7': [(0, 593, 15),
  (1, 133, 14),
  (2, 100, 12),
  (3, 140, 12),
  (4, 137, 12),
  (5, 149, 12),
  (6, 1, 12),
  (7, 35, 12),
  (8, 1, 12),
  (9, 1, 12),
  (10, 1, 12),
  (11, 1, 12)],
 'q8': [(0, 593, 20),
  (1, 133, 15),
  (2, 1, 12),
  (3, 100, 12),
  (4, 200, 12),
  (5, 100, 12),
  (6

In [28]:
def compute_resource_space(data):
    resource_space = {}
    for query_id, stages in data.items():
        resource_space[query_id] = sum(num_tasks * runtime for _, num_tasks, runtime in stages)
    return resource_space

In [29]:
query_resource_requirements = compute_resource_space(result)

In [30]:
query_resource_requirements

{'q1': 10698,
 'q2': 7868,
 'q3': 17471,
 'q4': 13659,
 'q5': 20436,
 'q6': 8314,
 'q7': 17549,
 'q8': 23395,
 'q9': 23138,
 'q10': 15133,
 'q11': 4924,
 'q12': 10680,
 'q13': 5400,
 'q14': 10020,
 'q15': 12528,
 'q16': 8352,
 'q17': 17760,
 'q18': 21160,
 'q19': 8328,
 'q20': 12704,
 'q21': 34128,
 'q22': 5978}

In [31]:
def bucketize_queries(resource_space):
    buckets = {"easy": [], "medium": [], "hard": []}
    for query_id, value in resource_space.items():
        if value < 10000:
            buckets["easy"].append(query_id)
        elif 10000 <= value <= 20000:
            buckets["medium"].append(query_id)
        else:
            buckets["hard"].append(query_id)
    return buckets

In [32]:
buckets = bucketize_queries(query_resource_requirements)

In [33]:
buckets

{'easy': ['q2', 'q6', 'q11', 'q13', 'q16', 'q19', 'q22'],
 'medium': ['q1', 'q3', 'q4', 'q7', 'q10', 'q12', 'q14', 'q15', 'q17', 'q20'],
 'hard': ['q5', 'q8', 'q9', 'q18', 'q21']}

In [ ]:
# Retry with rectangular mapping?